# Cleaned code assignment

Refactored version of the uploaded notebook code.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


DATA_DIR = Path("Data")
EUROSTAT_FILE = "Eurostat_employment_isco.xlsx"
TASKS_FILE = "onet_tasks.csv"
ISCO_LEVELS = range(1, 10)
COUNTRIES = ["Belgium", "Poland", "Spain"]
TASK_VARS = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]
INDEX_NAME = "NRCA"
PLOT_STEP = 3


def weighted_standardize(values: pd.Series, weights: pd.Series) -> pd.Series:
    """Return a weighted z-score series."""
    mean_value = np.average(values, weights=weights)
    variance = np.average((values - mean_value) ** 2, weights=weights)
    std_dev = np.sqrt(variance)

    if std_dev == 0:
        return pd.Series(np.zeros(len(values)), index=values.index)

    return (values - mean_value) / std_dev


def load_employment_data(
    data_dir: Path,
    countries: list[str],
    isco_levels: range,
    eurostat_file: str,
) -> pd.DataFrame:
    """Load employment data, add ISCO codes, totals and occupation shares."""
    frames = []

    for level in isco_levels:
        df = pd.read_excel(data_dir / eurostat_file, sheet_name=f"ISCO{level}")
        df = df.copy()
        df["ISCO"] = level
        frames.append(df)

    all_data = pd.concat(frames, ignore_index=True)

    for country in countries:
        total_col = f"total_{country}"
        share_col = f"share_{country}"

        country_totals = sum(frame[country] for frame in frames)
        all_data[total_col] = pd.concat(
            [country_totals] * len(list(isco_levels)),
            ignore_index=True,
        )
        all_data[share_col] = all_data[country] / all_data[total_col]

    return all_data


def load_task_data(data_dir: Path, tasks_file: str) -> pd.DataFrame:
    """Load and aggregate O*NET task data to 1-digit ISCO level."""
    task_data = pd.read_csv(data_dir / tasks_file)
    task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[0].astype(int)

    aggdata = task_data.groupby("isco08_1dig").mean(numeric_only=True)
    aggdata = aggdata.drop(columns=["isco08"], errors="ignore")

    return aggdata


def add_standardized_task_scores(
    combined: pd.DataFrame,
    countries: list[str],
    task_vars: list[str],
) -> pd.DataFrame:
    """Standardize each task variable separately for each country."""
    for country in countries:
        weight_col = f"share_{country}"
        for task in task_vars:
            std_col = f"std_{country}_{task}"
            combined[std_col] = weighted_standardize(
                combined[task],
                combined[weight_col],
            )

    return combined


def calculate_country_index(
    combined: pd.DataFrame,
    countries: list[str],
    task_vars: list[str],
    index_name: str,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Build a composite index for each country and aggregate it over time."""
    aggregated = {}

    for country in countries:
        weight_col = f"share_{country}"
        component_cols = [f"std_{country}_{task}" for task in task_vars]
        raw_index_col = f"{country}_{index_name}"
        std_index_col = f"std_{country}_{index_name}"
        weighted_index_col = f"multip_{country}_{index_name}"

        combined[raw_index_col] = combined[component_cols].sum(axis=1)
        combined[std_index_col] = weighted_standardize(
            combined[raw_index_col],
            combined[weight_col],
        )
        combined[weighted_index_col] = combined[std_index_col] * combined[weight_col]

        aggregated[country] = (
            combined.groupby("TIME", as_index=False)[weighted_index_col].sum()
        )

    return combined, aggregated


def plot_country_series(
    aggregated: dict[str, pd.DataFrame],
    countries: list[str],
    index_name: str,
    plot_step: int = 3,
) -> None:
    """Plot the final time series for each country."""
    for country in countries:
        value_col = f"multip_{country}_{index_name}"
        series = aggregated[country]

        plt.figure()
        plt.plot(series["TIME"], series[value_col])
        plt.title(f"{country} {index_name} over time")
        plt.xticks(
            range(0, len(series), plot_step),
            series["TIME"].iloc[::plot_step],
            rotation=45,
        )
        plt.tight_layout()
        plt.show()


def main() -> None:
    all_data = load_employment_data(
        data_dir=DATA_DIR,
        countries=COUNTRIES,
        isco_levels=ISCO_LEVELS,
        eurostat_file=EUROSTAT_FILE,
    )

    task_data_agg = load_task_data(
        data_dir=DATA_DIR,
        tasks_file=TASKS_FILE,
    )

    combined = all_data.merge(
        task_data_agg,
        left_on="ISCO",
        right_index=True,
        how="left",
    )

    combined = add_standardized_task_scores(
        combined=combined,
        countries=COUNTRIES,
        task_vars=TASK_VARS,
    )

    combined, aggregated = calculate_country_index(
        combined=combined,
        countries=COUNTRIES,
        task_vars=TASK_VARS,
        index_name=INDEX_NAME,
    )

    plot_country_series(
        aggregated=aggregated,
        countries=COUNTRIES,
        index_name=INDEX_NAME,
        plot_step=PLOT_STEP,
    )


if __name__ == "__main__":
    main()
